# Fine-tune a lightweight open-source LLM on your company PDF

End-to-end, Colab-friendly (free **T4 GPU** is enough):

1. **Extract** text from the PDF and chunk it
2. **Build a Q&A dataset** from the PDF (heuristic or LLM-generated)
3. **Baseline**: how does the untrained model do? And with **RAG**?
4. **Fine-tune** with **SFT** using **LoRA / QLoRA / DoRA / full** fine-tuning
5. **Evaluate** accuracy (fact recall, F1, ROUGE-L, semantic similarity, LLM judge, refusal rate)
6. **Chat** with the model, merge & export
7. Optional: **CPT -> SFT**, **RAFT**, **DPO**

Default model: `Qwen/Qwen2.5-0.5B-Instruct` (0.5B params, Apache-2.0). See the README for the theory and the comparison tables.

> Colab: *Runtime -> Change runtime type -> T4 GPU*. Heavy steps run as `!python src/...` so GPU memory is freed after each step.

## 0. Setup

In [ ]:
import os, sys, json
if os.path.exists("../src/common.py"):          # opened from notebooks/
    os.chdir("..")
if not os.path.exists("src/common.py"):         # Colab: clone the repo
    if not os.path.exists("llmfinetuning"):
        !git clone https://github.com/sifayathf/llmfinetuning.git
    os.chdir("llmfinetuning")
!pip -q install -r requirements.txt
sys.path.insert(0, "src")

In [ ]:
import torch
print("CUDA:", torch.cuda.is_available(), torch.cuda.get_device_name(0) if torch.cuda.is_available() else "")

### Configuration
* `BASE_MODEL` - any small chat model: `Qwen/Qwen2.5-0.5B-Instruct`, `Qwen/Qwen3-0.6B`, `HuggingFaceTB/SmolLM2-360M-Instruct`, `Qwen/Qwen2.5-1.5B-Instruct`, `meta-llama/Llama-3.2-1B-Instruct` (gated: needs HF login)
* `METHOD` - `lora` (default), `qlora` (least memory, GPU only), `dora`, `full`

In [ ]:
BASE_MODEL = "Qwen/Qwen2.5-0.5B-Instruct"
METHOD = "lora"
PDF_PATH = "data/sample_troubleshooting_guide.pdf"   # replace with your PDF
os.environ["BASE_MODEL"] = BASE_MODEL

## 1. Get the PDF
Use the bundled fictional call-center troubleshooting guide, or upload your own (Colab).

In [ ]:
if not os.path.exists(PDF_PATH):
    !python scripts/make_sample_pdf.py

# --- Upload your own PDF in Colab (uncomment) ---
# from google.colab import files
# up = files.upload(); PDF_PATH = "data/" + list(up)[0]; os.rename(list(up)[0], PDF_PATH)

## 2. Extract & chunk the text

In [ ]:
!python src/pdf_extract.py --pdf "{PDF_PATH}" --chunk-words 220 --overlap 40
from common import read_jsonl
chunks = read_jsonl("data/processed/chunks.jsonl")
print(len(chunks), "chunks\n"); print(chunks[1]["text"][:800])

## 3. Build the Q&A training set

An LLM learns facts best from **many question/answer pairs that express each fact in different ways** - not from raw text.

* `heuristic` - no LLM needed; parses headings and *Symptoms / Cause / Resolution* blocks. Great for structured troubleshooting guides.
* `llm` - **recommended for real documents**: a bigger *teacher* model writes Q&A pairs + paraphrases from every chunk (synthetic data / distillation). Use OpenAI, a local Ollama server, or a HF model.

One paraphrase of every fact is **held out for testing**; off-topic questions with an *"I don't know"* answer are added so the model learns to refuse instead of hallucinate.

In [ ]:
!python src/build_dataset.py --method heuristic

# --- Teacher LLM alternatives (better quality) ---
# Ollama running locally:  ollama pull qwen2.5:7b-instruct
# !OPENAI_API_KEY=ollama python src/build_dataset.py --method llm --backend openai --base-url http://localhost:11434/v1 --teacher qwen2.5:7b-instruct
# OpenAI:
# os.environ["OPENAI_API_KEY"] = "sk-..."
# !python src/build_dataset.py --method llm --backend openai --teacher gpt-4o-mini
# Local HF teacher on the GPU:
# !python src/build_dataset.py --method llm --backend hf --teacher Qwen/Qwen2.5-3B-Instruct

In [ ]:
import pandas as pd
train = pd.DataFrame(read_jsonl("data/processed/train.jsonl")); test = pd.DataFrame(read_jsonl("data/processed/test.jsonl"))
print(len(train), "train /", len(test), "test"); print(test["split"].value_counts())
pd.set_option("display.max_colwidth", 120); train[["question", "answer", "type"]].head(8)

## 4. Baselines - before any training
Always measure the base model first, otherwise you can't prove fine-tuning helped.
`--sim-model none` skips the semantic-similarity metric if you want it faster.

In [ ]:
!python src/eval_model.py --model {BASE_MODEL} --name 1-base

### RAG baseline (no training)
Retrieve the top-k chunks for each question and put them in the prompt.

In [ ]:
!python src/rag.py build                     # BAAI/bge-small-en-v1.5 embeddings ("--embedder tfidf" = fully offline)
!python src/eval_model.py --model {BASE_MODEL} --rag-index outputs/rag_index --name 2-base+rag

## 5. Fine-tune (SFT)

| method | what is trained | GPU memory (0.5B) | notes |
|---|---|---|---|
| `lora`  | ~1% extra adapter weights | ~3-4 GB | best default |
| `qlora` | LoRA on 4-bit frozen base | ~2 GB | needs NVIDIA GPU; for bigger models on small GPUs |
| `dora`  | LoRA + magnitude vector | ~4 GB | often slightly more accurate than LoRA, ~1.3x slower |
| `full`  | all weights | ~10-12 GB | most capacity, risk of forgetting; use lr 1e-5 - 2e-5 |

Knowledge injection needs **several epochs** (default 8 for adapters). Watch `eval_loss`: if it rises while `train_loss` keeps falling you are over-fitting - lower the epochs.

In [ ]:
!python src/train_sft.py --method {METHOD}
# other options:
# !python src/train_sft.py --method qlora
# !python src/train_sft.py --method full --epochs 3 --lr 2e-5
# !python src/train_sft.py --method lora --lora-r 64 --lora-alpha 128 --epochs 10   # more capacity

## 6. Evaluate the fine-tuned model (alone and + RAG)

In [ ]:
ADAPTER = f"outputs/sft-{METHOD}"
!python src/eval_model.py --model {ADAPTER} --name 3-sft-{METHOD}
!python src/eval_model.py --model {ADAPTER} --rag-index outputs/rag_index --name 4-sft-{METHOD}+rag

In [ ]:
!python src/eval_model.py --compare "results/*_summary.json"

**How to read the table**
* `accuracy_doc` - share of in-document questions answered correctly (judge >= 4, or >= 50% key facts recalled without refusing)
* `fact_recall` - share of key facts (numbers, codes, names, IPs) from the reference found in the answer
* `token_f1`, `rouge_l` - word overlap with the reference (ROUGE-L also rewards correct step order)
* `semantic_sim` - meaning similarity (0-1)
* `false_refusal_rate` - said *"I don't know"* although the answer is in the PDF (lower is better)
* `refusal_accuracy_oos` - correctly refused off-topic questions (higher is better)

Then **read the worst answers yourself** - automatic metrics are only a proxy:

In [ ]:
res = pd.read_csv(f"results/3-sft-{METHOD}.csv")
res.sort_values("fact_recall")[["split", "question", "reference", "prediction", "fact_recall", "correct"]].head(10)

### (Optional) LLM-as-a-judge
A stronger model scores each answer 1-5 against the reference - closest to human grading.

In [ ]:
# os.environ["OPENAI_API_KEY"] = "sk-..."
# !python src/eval_model.py --model outputs/sft-{METHOD} --judge openai --judge-model gpt-4o-mini --name 3-sft-{METHOD}-judged
# Local judge on GPU:
# !python src/eval_model.py --model outputs/sft-{METHOD} --judge hf --judge-model Qwen/Qwen2.5-7B-Instruct --load-in-4bit --name judged

## 7. Chat with your model

In [ ]:
from common import load_model_and_tokenizer, answer
model, tok = load_model_and_tokenizer(f"outputs/sft-{METHOD}")
for q in ["My router shows error E-307, what do I do?",
          "How much credit can a Tier 1 agent approve?",
          "What is the default router admin address?",
          "Who won the football world cup?"]:
    print("Q:", q); print("A:", answer(model, tok, q), "\n")

In [ ]:
# with RAG
from rag import Retriever
ret = Retriever("outputs/rag_index")
q = "The internet light is red, what should the customer do?"
print(answer(model, tok, q, ret.context(q, k=3)))
del model; torch.cuda.empty_cache()

## 8. Optional advanced methods

### 8a. Continued pre-training (CPT) -> SFT
First let the model read the raw PDF text (domain vocabulary, facts), then teach it Q&A.

In [ ]:
# !python src/train_cpt.py --method lora --epochs 3 --merge            # -> outputs/cpt-lora-merged
# !python src/train_sft.py --method lora --model outputs/cpt-lora-merged --output outputs/cpt-sft-lora
# !python src/eval_model.py --model outputs/cpt-sft-lora --name 5-cpt+sft

### 8b. RAFT (Retrieval-Augmented Fine-Tuning)
Train with the relevant chunk **plus distractor chunks** in the prompt so the model learns to read retrieved context and ignore noise. Always use it **with** RAG at inference.

In [ ]:
# !python src/train_sft.py --method lora --raft --output outputs/raft-lora
# !python src/eval_model.py --model outputs/raft-lora --rag-index outputs/rag_index --name 6-raft+rag

### 8c. DPO (preference tuning) after SFT
Pairs *(reference answer = chosen, model's wrong answer = rejected)* are built automatically; reduces remaining hallucinations.

In [ ]:
# !python src/train_sft.py --method lora --merge                      # -> outputs/sft-lora-merged
# !python src/train_dpo.py --model outputs/sft-lora-merged            # -> outputs/dpo-lora
# !python src/eval_model.py --model outputs/dpo-lora --name 7-sft+dpo

## 9. Merge & export
Merging the LoRA adapter gives a normal model folder you can serve with vLLM / TGI, or convert to GGUF for Ollama / llama.cpp:
```
python llama.cpp/convert_hf_to_gguf.py outputs/sft-lora-merged --outfile company-bot.gguf
```

In [ ]:
!python src/chat.py --model outputs/sft-{METHOD} --merge-to outputs/sft-{METHOD}-merged
# from google.colab import drive; drive.mount("/content/drive")
# !cp -r outputs/sft-{METHOD}-merged /content/drive/MyDrive/